# Defining the Teacher Forcing model layers

You will be defining a new-and-improved version of the machine translation model that you defined earlier. Did you know that models like the Google Machine Translator used this Teacher Forcing technique to train their model?

In [3]:
en_len = 20
en_vocab = 200
hsize = 64
fr_len = 25
fr_vocab = 250

In [4]:
# Import the layers submodule from keras
import tensorflow.keras.layers as layers

en_inputs = layers.Input(shape=(en_len, en_vocab))
en_gru = layers.GRU(hsize, return_state=True)
# Get the encoder output and state
en_out, en_state = en_gru(en_inputs)

# Define the decoder input layer
de_inputs = layers.Input(shape=(fr_len-1, fr_vocab))
de_gru = layers.GRU(hsize, return_sequences=True)
de_out = de_gru(de_inputs, initial_state=en_state)
# Define a TimeDistributed Dense softmax layer with fr_vocab nodes
de_dense = layers.TimeDistributed(layers.Dense(fr_vocab, activation="softmax"))
de_pred = de_dense(de_out)

# Defining the Teacher Forcing model

With all the layers created, the next step would be to define a Keras Model object. This model is slightly different to the one that you defined earlier, as the new model has two input layers.

In [6]:
# Import the Keras Model object
from tensorflow.keras.models import Model

# Define a model
nmt_tf = Model(inputs=[en_inputs, de_inputs], outputs=de_pred)
# Compile the model with optimizer and loss
nmt_tf.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
# Print the summary of the model
nmt_tf.summary()

Model: "functional_3"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 20, 200)]    0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 24, 250)]    0                                            
__________________________________________________________________________________________________
gru (GRU)                       [(None, 64), (None,  51072       input_1[0][0]                    
__________________________________________________________________________________________________
gru_1 (GRU)                     (None, 24, 64)       60672       input_2[0][0]                    
                                                                 gru[0][1]             

# Preprocessing data

You now need to process the data for our new model which has two inputs and a single output. The two inputs are, the one-hot encoded English words and one-hot encoded French words excluding the last word.

The output would be the one-hot encoded French words excluding the first word. In other words, in the decoder, each input French word has an output, which is the next word. Here you will learn how to implement that.

In [7]:
import pandas as pd
en_text = pd.read_csv('dataset/vocab_en.txt', header=None, delimiter='\n')
# en_text.head()
fr_text = pd.read_csv('dataset/vocab_fr.txt', header=None, delimiter='\n')
# fr_text.head()
en_text = en_text.iloc[:,0].values.tolist()
fr_text = fr_text.iloc[:,0].values.tolist()


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

en_tok = Tokenizer(num_words=200, oov_token='UNK') 
en_tok.fit_on_texts(en_text)

fr_tok = Tokenizer(num_words=250, oov_token='UNK') 
fr_tok.fit_on_texts(fr_text)


In [9]:
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical

def sents2seqs(input_type, sentences, onehot=False, pad_type='post', reverse=False):
    assert input_type in ["source", "target"]
    if input_type == 'source':
      tokenizer = en_tok
      pad_length = en_len
      vocab_size = en_vocab
    elif input_type == 'target':
      tokenizer = fr_tok
      pad_length = fr_len
      vocab_size = fr_vocab
    
    encoded_text = tokenizer.texts_to_sequences(sentences)
    preproc_text = pad_sequences(encoded_text, padding=pad_type, truncating='post', maxlen=en_len)
    if reverse:
      preproc_text = preproc_text[:,::-1]
      
    if onehot:
        assert vocab_size is not None, "Cannot do to_categorical without num_classes for safety"
        preproc_text = to_categorical(preproc_text, num_classes=vocab_size)
    return preproc_text

In [10]:
import numpy as np
bsize = 250
for i in range(0, len(en_text), bsize):
  # Get the encoder inputs using the sents2seqs() function
  en_x = sents2seqs('source', en_text[i:i+bsize], onehot=True, reverse=True)
  # Get the decoder inputs/outputs using the sents2seqs() function
  de_xy = sents2seqs('target', fr_text[i:i+bsize], onehot=True)
  # Separate the decoder inputs from de_xy
  de_x = de_xy[:,:-1,:]
  # Separate the decoder outputs from de_xy
  de_y = de_xy[:,1:,:]
  
  print("Data from ", i, " to ", i+bsize)
  print("\tnp.argmax() => en_x[0]: ", np.argmax(en_x[0], axis=-1))
  print("\tnp.argmax() => de_x[0]: ", np.argmax(de_x[0], axis=-1))
  print("\tnp.argmax() => de_y[0]: ", np.argmax(de_y[0], axis=-1))

Data from  0  to  250
	np.argmax() => en_x[0]:  [ 0  0  0  0  0  0  0 45  3 56  2  4  8 40  5 68  9  2 24 18]
	np.argmax() => de_x[0]:  [ 36  35   2   9  68  38  12  25   7   4   2 113   3  51   0   0   0   0
   0]
	np.argmax() => de_y[0]:  [ 35   2   9  68  38  12  25   7   4   2 113   3  51   0   0   0   0   0
   0]
Data from  250  to  500
	np.argmax() => en_x[0]:  [ 0  0  0  0  0  0  0 62  3 66 11  2  4  8 41  5 52  9  2 25]
	np.argmax() => de_x[0]:  [ 8 34  2  9 11 82  3 48  7  4  2 10 61  3 57  0  0  0  0]
	np.argmax() => de_y[0]:  [34  2  9 11 82  3 48  7  4  2 10 61  3 57  0  0  0  0  0]
Data from  500  to  750
	np.argmax() => en_x[0]:  [ 0  0  0  0  0  0  0 49  3 54  9  2  4  7 44  5 64 11  2 23]
	np.argmax() => de_x[0]:  [102   2  10  67   3  50   6   4   2   9  69   3  54   0   0   0   0   0
   0]
	np.argmax() => de_y[0]:  [ 2 10 67  3 50  6  4  2  9 69  3 54  0  0  0  0  0  0  0]
Data from  750  to  1000
	np.argmax() => en_x[0]:  [ 0  0  0  0  0 38  3 52 11  2  4  7 41  5 54

# Training the model 

Did you know that in 2017, Google translate served more than 500 million users daily?

Here, you will train your first Teacher Forced model. Teacher Forcing is commonly used in sequence to sequence models like your neural machine translator to achieve better performance.

In [11]:
data_size = 1500
n_epochs, bsize = 3, 250

for ei in range(n_epochs):
  for i in range(0,data_size,bsize):
    en_x = sents2seqs('source', en_text[i:i+bsize], onehot=True, reverse=True)
    de_xy = sents2seqs('target', fr_text[i:i+bsize], onehot=True)
    # Separate the decoder inputs from de_xy
    de_x = de_xy[:,:-1,:]
    # Separate the decoder outputs from de_xy
    de_y = de_xy[:,1:,:]
    # Train the model on a single batch of data    
    nmt_tf.train_on_batch([en_x,de_x], de_y)    
    # Obtain the eval metrics for the training data
    res = nmt_tf.evaluate([en_x,de_x], de_y, batch_size=bsize, verbose=0)
    print("{} => Train Loss:{}, Train Acc: {}".format(ei+1,res[0], res[1]*100.0))  

1 => Train Loss:5.5052080154418945, Train Acc: 0.27368420269340277
1 => Train Loss:5.494729042053223, Train Acc: 0.7157894782721996
1 => Train Loss:5.483461380004883, Train Acc: 1.3894736766815186
1 => Train Loss:5.470432758331299, Train Acc: 2.9052631929516792
1 => Train Loss:5.4619011878967285, Train Acc: 6.084210425615311
1 => Train Loss:5.447921276092529, Train Acc: 31.22105300426483
2 => Train Loss:5.433997631072998, Train Acc: 42.378947138786316
2 => Train Loss:5.423848628997803, Train Acc: 42.44210422039032
2 => Train Loss:5.4109625816345215, Train Acc: 42.65263080596924
2 => Train Loss:5.392319202423096, Train Acc: 45.05263268947601
2 => Train Loss:5.384315013885498, Train Acc: 42.589473724365234
2 => Train Loss:5.364482879638672, Train Acc: 43.59999895095825
3 => Train Loss:5.344571113586426, Train Acc: 44.252631068229675
3 => Train Loss:5.331023693084717, Train Acc: 43.15789341926575
3 => Train Loss:5.311923503875732, Train Acc: 42.35789477825165
3 => Train Loss:5.28107833862

# Splitting training and validation data

You will be creating training and validation datasets. Keeping a validation dataset and monitoring the performance of model on that set is a good practice to avoid overfitting.

In [12]:
train_size, valid_size = 800, 200
# Define a sequence of indices from 0 to size of en_text
inds = np.arange(len(en_text))
np.random.shuffle(inds)
# Define train_inds as first train_size indices
train_inds = inds[:train_size]
valid_inds = inds[train_size:train_size+valid_size]
# Define tr_en (train EN sentences) and tr_fr (train FR sentences)
tr_en = [en_text[ti] for ti in train_inds]
tr_fr = [fr_text[ti] for ti in train_inds]
# Define v_en (valid EN sentences) and v_fr (valid FR sentences)
v_en = [en_text[vi] for vi in valid_inds]
v_fr = [fr_text[vi] for vi in valid_inds]
print('Training (EN):\n', tr_en[:3], '\nTraining (FR):\n', tr_fr[:3])
print('\nValid (EN):\n', v_en[:3], '\nValid (FR):\n', v_fr[:3])

Training (EN):
 ['oranges are her least favorite fruit .', 'france is never cold during july , but it is sometimes nice in summer .', 'new jersey is never chilly during autumn , but it is sometimes warm in october .'] 
Training (FR):
 ['oranges sont son fruit préféré moins .', 'france ne fait jamais froid en juillet , mais il est parfois agréable en été .', "new jersey est jamais froid au cours de l' automne , mais il est parfois chaud en octobre ."]

Valid (EN):
 ['their least liked fruit is the grapefruit , but our least liked is the pear .', 'we like bananas , oranges , and strawberries .', 'france is usually beautiful during winter , but it is sometimes freezing in april .'] 
Valid (FR):
 ['leurs fruits moins aimé est le pamplemousse , mais notre moins aimé est la poire .', 'nous aimons les bananes , les oranges et les fraises .', "la france est généralement beau pendant l' hiver , mais il est parfois le gel en avril ."]


# Training the model with validation

Here you will train the model using Teacher Forcing and also perform a validation step. You will train the model for multiple epochs and multiple iterations. Then at the end of each epoch, you will run the validation step and obtain the results.

In [13]:
for ei in range(n_epochs):
  for i in range(0,train_size,bsize):    
    en_x = sents2seqs('source', tr_en[i:i+bsize], onehot=True, reverse=True)
    de_xy = sents2seqs('target', tr_fr[i:i+bsize], onehot=True)
    # Create a single batch of decoder inputs and outputs
    de_x, de_y = de_xy[:,:-1,:], de_xy[:,1:,:]
    # Train the model on a single batch of data
    nmt_tf.train_on_batch([en_x,de_x], de_y)      
  v_en_x = sents2seqs('source', v_en, onehot=True, reverse=True)
  # Create a single batch of validation decoder inputs and outputs
  v_de_xy = sents2seqs('target', v_fr, onehot=True)
  v_de_x, v_de_y = v_de_xy[:,:-1,:], v_de_xy[:,1:,:]
  # Evaluate the trained model on the validation data
  res = nmt_tf.evaluate([v_en_x,v_de_x], v_de_y, batch_size=valid_size, verbose=0)
  print("{} => Loss:{}, Val Acc: {}".format(ei+1,res[0], res[1]*100.0))

1 => Loss:5.076735019683838, Val Acc: 42.47368276119232
2 => Loss:4.756934642791748, Val Acc: 42.1842098236084
3 => Loss:4.026854515075684, Val Acc: 41.63157939910889


# Defining the decoder of the inference model

The inference model is the model that will be used out in the wild to perform translations when required by the user. In this exercise, you will need to implement the decoder of the inference model.

The inference model decoder is different to the decoder of the training model. We can't feed the decoder with French words because that is what we want to predict. Luckily, there is a solution. We can use the predicted French word from the previous time step to feed the inference model decoder. Therefore, when you want to generate a translation, the decoder needs to generate one word at a time, while consuming the previous output as an input.

<center><img src="images/04.06.png"  style="width: 400px, height: 300px;"/></center>

In [14]:
import tensorflow.keras.layers as layers
from tensorflow.keras.models import Model
# Define an input layer that accepts a single onehot encoded word
de_inputs = layers.Input(shape=(1, fr_vocab))
# Define an input to accept the t-1 state
de_state_in = layers.Input(shape=(hsize,))
de_gru = layers.GRU(hsize, return_state=True)
# Get the output and state from the GRU layer
de_out, de_state_out = de_gru(de_inputs, initial_state=de_state_in)
de_dense = layers.Dense(fr_vocab, activation='softmax')
de_pred = de_dense(de_out)

# Define a model
decoder = Model(inputs=[de_inputs, de_state_in], outputs=[de_pred, de_state_out])
print(decoder.summary())

Model: "functional_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 1, 250)]     0                                            
__________________________________________________________________________________________________
input_4 (InputLayer)            [(None, 64)]         0                                            
__________________________________________________________________________________________________
gru_2 (GRU)                     [(None, 64), (None,  60672       input_3[0][0]                    
                                                                 input_4[0][0]                    
__________________________________________________________________________________________________
dense_1 (Dense)                 (None, 250)          16250       gru_2[0][0]           

# Link between the trained and inference model

Here you will be transferring the trained weights from the trained model to the inference model. In the encoder decoder model, there are three layers with parameters. They are,

- The encoder GRU layer
- The decoder GRU layer
- The decoder Dense layer
- The other layers, such as TimeDistributed do not have any parameters, thus don't require the copying of weights.

In [15]:
# # Load the weights to the encoder GRU from the trained model
# en_gru_w = tr_en_gru.get_weights()
# # Set the weights of the encoder GRU of the inference model
# en_gru.set_weights(en_gru_w)
# # Load and set the weights to the decoder GRU
# de_gru.set_weights(tr_de_gru.get_weights())
# # Load and set the weights to the decoder Dense
# de_dense.set_weights(tr_de_dense.get_weights())

# Generating translations

You will now be generating French translations using an inference model trained using Teacher Forcing.

This model (nmt_tf) has been trained for 50 epochs on 100,000 sentences which achieved around 98% accuracy on a 35000+ validation set. It might take longer for this exercise to initialize as the trained model needs to be loaded. 

In [17]:
def word2onehot(tokenizer, word, vocab_size):
    de_seq = tokenizer.texts_to_sequences([[word]])
    de_onehot = to_categorical(de_seq, num_classes=vocab_size)
    de_onehot = np.expand_dims(de_onehot, axis=1)    
    return de_onehot

def probs2word(probs, tok):
    wid = np.argmax(probs[0,:], axis=-1)
    w = tok.index_word[wid]
    return w

In [18]:
from keras.layers import Input, GRU, TimeDistributed, Dense
from keras.models import Model

# Define input shapes
input_shape_encoder = (15, 150)  # Shape of encoder input
input_shape_decoder = (19, 300)  # Shape of decoder input

# Define encoder input layer
encoder_input = Input(shape=input_shape_encoder, name='input_1')

# Define decoder input layer
decoder_input = Input(shape=input_shape_decoder, name='input_2')

# Define encoder GRU layer
encoder_gru, encoder_state = GRU(48, return_state=True, name='gru')(encoder_input)

# Define decoder GRU layer
decoder_gru = GRU(48, return_sequences=True, name='gru_1')(decoder_input, initial_state=encoder_state)

# Define time distributed layer
time_distributed = TimeDistributed(Dense(300), name='time_distributed')(decoder_gru)

# Create model
model = Model(inputs=[encoder_input, decoder_input], outputs=time_distributed)

# Print model summary
model.summary()


Model: "functional_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 15, 150)]    0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 19, 300)]    0                                            
__________________________________________________________________________________________________
gru (GRU)                       [(None, 48), (None,  28800       input_1[0][0]                    
__________________________________________________________________________________________________
gru_1 (GRU)                     (None, 19, 48)       50400       input_2[0][0]                    
                                                                 gru[0][1]             

In [19]:
from keras.layers import Input, GRU
from keras.models import Model

# Define input shape
input_shape_encoder = (15, 150)  # Shape of encoder input

# Define encoder input layer
encoder_input = Input(shape=input_shape_encoder, name='input_1')

# Define encoder GRU layer
encoder_gru, encoder_state = GRU(48, return_state=True, name='gru')(encoder_input)

# Create encoder model
encoder = Model(inputs=encoder_input, outputs=[encoder_gru, encoder_state])

# Print encoder model summary
encoder.summary()


Model: "functional_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 15, 150)]         0         
_________________________________________________________________
gru (GRU)                    [(None, 48), (None, 48)]  28800     
Total params: 28,800
Trainable params: 28,800
Non-trainable params: 0
_________________________________________________________________


In [21]:
# from keras.layers import Input, GRU, Dense
# from keras.models import Model

# # Define input shapes
# input_shape_decoder = (1, 300)  # Shape of decoder input
# input_shape_state = (None, 48)   # Shape of encoder state input

# # Define decoder input layers
# decoder_input = Input(shape=input_shape_decoder, name='input_2')
# state_input = Input(shape=input_shape_state, name='input_3')

# # Define decoder GRU layer
# decoder_gru, decoder_state = GRU(48, return_sequences=True, return_state=True, name='gru_1')(decoder_input, initial_state=state_input)

# # Define dense layer
# dense_output = Dense(300, name='dense')(decoder_gru)

# # Create decoder model
# decoder = Model(inputs=[decoder_input, state_input], outputs=dense_output)

# # Print decoder model summary
# decoder.summary()


In [23]:
# en_sent = ['the united states is sometimes chilly during december , but it is sometimes freezing in june .']
# print('English: {}'.format(en_sent))
# en_seq = sents2seqs('source', en_sent, onehot=True, reverse=True)
# # Predict the initial decoder state with the encoder
# de_s_t = encoder.predict(en_seq)
# de_seq = word2onehot(fr_tok, 'sos', fr_vocab)
# fr_sent = ''
# for i in range(fr_len):    
#   # Predict from the decoder and recursively assign the new state to de_s_t
#   de_prob, de_s_t = decoder.predict([de_seq,de_s_t])
#   # Get the word from the probability output using probs2word
#   de_w = probs2word(de_prob, fr_tok)
#   # Convert the word to a onehot sequence using word2onehot
#   de_seq = word2onehot(fr_tok, de_w, fr_vocab)
#   if de_w == 'eos': break
#   fr_sent += de_w + ' '
# print("French (Ours): {}".format(fr_sent))
# print("French (Google Translate): les etats-unis sont parfois froids en décembre, mais parfois gelés en juin")

# Measuring word vector similarity

In this lesson we will understand the power of word vectors using real world trained word vectors. These are word vectors extracted from a list of word vectors published by the Stanford NLP group. A word vector is a sequence or a vector of numerical values. For example, dog = (0.31, 0.92, 0.13)

The distance between word vectors can be measured using a pair-wise similarity metric. Here we will be using sklearn.metrics.pairwise.cosine_similarity. Cosine similarity produces a higher values when the element-wise similarity of two vectors is high and vice-versa.

In [24]:
cat_vector = np.array([[ 0.39394888, -0.26333705,  0.08621896,  0.01091912, -0.32225883,
        -0.10658745, -0.25810188,  0.35068503, -0.21361879,  0.04625478,
         0.27992395,  0.34801784, -0.16187142, -0.52442133,  0.305441  ,
         0.2697149 ,  0.13099045, -0.13699648,  0.22602458,  0.4918219 ,
        -0.321026  ,  0.0057017 , -0.04981493, -0.02993789,  0.29878485,
        -0.3817054 , -0.05704023, -0.23727393,  0.08146162, -0.40114117,
         0.15975171,  0.23145257,  0.3185359 , -0.18709224, -0.2787781 ,
         0.21370722,  0.00497514,  0.04430564, -0.02908332, -0.17749819,
        -0.30718255, -0.25935414,  0.10115459,  0.01766969, -0.16098014,
         0.3519572 , -0.0734664 , -0.3723043 ,  0.37262923, -0.34300864,
        -0.07277397, -0.00659163,  0.23748   ,  0.13399196,  0.5749845 ,
         0.18539067,  0.20504089, -0.3285161 ,  0.12926023, -0.3795821 ,
         0.3177269 ,  0.03064232,  0.1554114 , -0.00926715,  0.02937708,
         0.16277929,  0.392671  ,  0.53800666, -0.45575112,  0.38556987,
         0.06706808, -0.3802721 ,  0.03560793, -0.46856695, -0.2644591 ,
        -0.14335115,  0.10971718, -0.06499432,  0.26991028, -0.32314897,
        -0.07378025,  0.3670569 ,  0.15895942,  0.4204117 ,  0.00254397,
        -0.31696656,  0.04942322,  0.1372362 ,  0.4173302 , -0.29748875,
        -0.0426416 ,  0.2565474 , -0.3946782 ,  0.0325617 ,  0.15673377,
         0.38802406]])
window_vector = np.array([[ 0.13325554,  0.14915967, -0.30721307,  0.09023898, -0.14277737,
        -0.3429877 ,  0.6591184 ,  0.22567806,  0.50057876, -0.14418888,
         0.7266859 ,  0.31579646, -0.04160513,  0.21206841,  0.53187066,
         0.04340999, -0.2023547 , -0.28836393, -0.03026843,  0.09326456,
        -0.15379508,  0.41042185, -0.3575508 ,  0.33814973, -0.03736347,
        -0.5686376 , -0.35828388, -0.09262004, -0.04406314, -0.30181015,
         0.02286413,  0.30441186,  0.05470408, -0.71816754,  0.13778108,
        -0.16295761,  0.09613513, -0.55940634, -0.57205784, -0.7506126 ,
        -0.31361568,  0.13012886,  0.57220346, -0.13439907, -0.22363257,
         0.42024654,  0.01096254, -0.2432713 ,  0.5192242 ,  0.28016105,
        -0.4941133 ,  0.39157686,  0.119226  , -0.35288376,  0.05466123,
        -0.0957669 , -0.0585629 ,  0.20194787, -0.03544503,  0.12973273,
         0.12909019,  0.03881926,  0.41618073, -0.31265515,  0.40158305,
         0.16749044, -0.25229704, -0.18045057, -0.11000848,  0.3967997 ,
         0.71698666, -0.7267666 , -0.08943647, -0.90438217, -0.07253361,
         0.00425288,  0.12160529,  0.02097287,  0.23188019, -0.22092348,
        -0.07865301,  0.27505714, -0.38818485, -0.37019807,  0.3313728 ,
        -0.03597212,  0.12701605, -0.7077018 ,  0.7482011 ,  0.24647832,
        -0.46840447, -0.314954  , -0.85316503, -0.2805526 , -0.08966991,
         0.5265477 ]])
dog_vector = np.array([[ 0.3996625 , -0.30058518,  0.04715299, -0.05910603, -0.11148381,
         0.1093993 ,  0.01090461,  0.23533289,  0.23022532, -0.10736388,
         0.27194744,  0.0234511 , -0.31267357, -0.3329223 ,  0.5383476 ,
         0.01967983, -0.01314637, -0.47769648,  0.21548632,  0.09172583,
        -0.21391751,  0.06045289, -0.33278054, -0.03726481,  0.3942353 ,
        -0.22838424, -0.14873335, -0.2946241 , -0.30747628,  0.01201936,
         0.3984858 ,  0.17620644,  0.4095585 , -0.27740127, -0.01438677,
        -0.12329695, -0.04173272, -0.10615015, -0.07224224, -0.48236308,
        -0.53165466, -0.22681382, -0.04323539,  0.36861056,  0.12315456,
         0.11744846,  0.10699135, -0.23141243,  0.61972225, -0.3322037 ,
        -0.33399817,  0.14830801,  0.5658622 , -0.11296458,  0.16682227,
        -0.09137795,  0.10273071, -0.25096542,  0.29488757, -0.2795514 ,
         0.30368486, -0.27580926, -0.14012972, -0.24056736,  0.24908563,
         0.14914584, -0.11725031,  0.24276581, -0.00545608,  0.45917222,
         0.03866611, -0.38174465, -0.36502177, -0.4143763 , -0.1863586 ,
        -0.45668072,  0.3324825 ,  0.12192825,  0.47594503, -0.18364649,
         0.42104712, -0.01100355, -0.18209179,  0.28632256, -0.00409962,
        -0.394309  , -0.00906276, -0.23715773,  0.3950311 , -0.5642396 ,
         0.42861956, -0.46919912, -0.3660173 , -0.05302884,  0.11614416,
         0.03718512]])

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

# Print the length of the cat_vector
print('Length of the cat_vector: ', cat_vector.size)

# Compute and print the similarity between cat and window vectors
dist_cat_window = cosine_similarity(cat_vector, window_vector)
print('Similarity(cat, window): ', dist_cat_window)

# Compute and print the similarity between cat and dog vectors
print('Similarity(cat,dog): ', cosine_similarity(cat_vector, dog_vector))

Length of the cat_vector:  96
Similarity(cat, window):  [[0.32330843]]
Similarity(cat,dog):  [[0.60183563]]


# Defining the embedding model

You will be defining a Keras model that:

- Uses Embedding layers
- Will be trained with Teacher Forcing

This model will have two embedding layers; an encoder embedding layer and a decoder embedding layer. Furthermore, as the model is trained using Teacher Forcing, it will use a sequence length of fr_len-1 in the decoder Input layer.

In [ ]:
en_vocab = 200

In [29]:
from keras.layers import Embedding
# Define an input layer which accepts a sequence of word IDs
en_inputs = Input(shape=(en_len,))
# Define an Embedding layer which accepts en_inputs
en_emb = Embedding(en_vocab, 96, input_length=en_len)(en_inputs)
en_out, en_state = GRU(hsize, return_state=True)(en_emb)

de_inputs = Input(shape=(fr_len-1,))
# Define an Embedding layer which accepts de_inputs
de_emb = Embedding(fr_vocab, 96, input_length=fr_len-1)(de_inputs)
de_out, _ = GRU(hsize, return_sequences=True, return_state=True)(de_emb, initial_state=en_state)
de_pred = TimeDistributed(Dense(fr_vocab, activation='softmax'))(de_out)

# Define the Model which accepts encoder/decoder inputs and outputs predictions 
nmt_emb = Model([en_inputs, de_inputs], de_pred)
nmt_emb.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])

# Training the word embedding based model

Here you will learn how to implement the training process for a machine translator model that uses word embeddings. A word is represented as a single number instead of a one-hot encoded vector as you did in previous exercises. You will train the model for multiple epochs while traversing through the full dataset in batches.

In [30]:
for ei in range(3):
  for i in range(0, train_size, bsize):    
    en_x = sents2seqs('source', tr_en[i:i+bsize], onehot=False, reverse=True)
    # Get a single batch of French sentences with no onehot encoding
    de_xy = sents2seqs('target', tr_fr[i:i+bsize], onehot=False)
    # Get all words except the last word in that batch
    de_x = de_xy[:,:-1]
    de_xy_oh = sents2seqs('target', tr_fr[i:i+bsize], onehot=True)
    # Get all words except the first from de_xy_oh
    de_y = de_xy_oh[:,1:,:]
    # Training the model on a single batch of data
    nmt_emb.train_on_batch([en_x,de_x], de_y)    
    res = nmt_emb.evaluate([en_x, de_x], de_y, batch_size=bsize, verbose=0)
    print("{} => Loss:{}, Train Acc: {}".format(ei+1,res[0], res[1]*100.0))

1 => Loss:5.512256622314453, Train Acc: 0.6105263251811266
1 => Loss:5.502020359039307, Train Acc: 17.157894372940063
1 => Loss:5.490842342376709, Train Acc: 38.42105269432068
1 => Loss:5.479250907897949, Train Acc: 41.894736886024475
2 => Loss:5.466843128204346, Train Acc: 42.778947949409485
2 => Loss:5.45570707321167, Train Acc: 41.789475083351135
2 => Loss:5.4411163330078125, Train Acc: 41.97894632816315
2 => Loss:5.4254560470581055, Train Acc: 42.42105185985565
3 => Loss:5.407793045043945, Train Acc: 42.73684322834015
3 => Loss:5.3922624588012695, Train Acc: 41.87368452548981
3 => Loss:5.3698954582214355, Train Acc: 42.31579005718231
3 => Loss:5.345460414886475, Train Acc: 42.31579005718231
